## pdfs ##

In [1]:
import pymupdf
import spacy
import re
import pandas as pd
import unicodedata
import os
from pathlib import Path

In [2]:
nlp = spacy.load('en_core_web_lg')

In [3]:
# word vectors (similarity) vs linguistic featues e.g. lemma e.g. transportation vs transport

renewable_energy = ['renewable', 'solar', 'wind', 'bioenergy', 'biofuel', 'biomass', 'hydropower', 'hydrogen', 'power', 'grid', 'transmission', 'generation']
energy_efficiency = ['efficiency', 'retrofit']
pollution_prevention_and_control = ['pollution', 'waste']
environmentally_sustainable_management_of_living_natural_resources_and_land_use = ['land', 'agriculture', 'forestry', 'forest', 'fisheries', 'food']
terrestrial_and_aquatic_biodiversity_conservation = ['terrestrial', 'aquatic', 'biodiversity', 'conservation']
clean_transportation = ['transportation', 'electric', 'battery', 'EV', 'charger', 'bus', 'rail', 'train', 'car', 'vehicle', 'bicycle', 'non-motorized', 'aviation']
sustainable_water_and_wastewater_management = ['water', 'potable', 'wastewater', 'sanitation', 'treatment']
climate_change_adaptation = ['adaptation', 'disaster']
circular_economy_and_or_ecoefficient_projects = ['circular', 'recycle', 'reuse']
green_buildings = ['buildings', 'appliances', 'housing']

In [25]:
# document = '/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/inscope framework reports/Climate_Bond_Framework_of_E1_SUBHOLDING_S_A__in_2021.pdf'

def find_uop(document, language):

    pdf = pymupdf.open(document)

    if language == 'EN':
        keywordsUOP = ['Use of Proceeds', 'Use of Funds', 'Use of the Proceeds']
        keywordsSEEGP = ['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Project Selection and Evaluation Process', 'Process for Project Evaluation and Selection', 'Project Selection and Assessment Process', 'Project Selection Criteria', 'Project evaluation & selection', 'Project Selection Process']

    elif language == 'PT':
        keywordsUOP = ['Uso de Recursos']
        keywordsSEEGP = ['Processo de Avaliação e Seleção de Projetos']

    elif language == 'ES':
        keywordsUOP = ['Uso de fondos', 'Uso de los fondos']
        keywordsSEEGP = ['XYZ']

    areaUOP = None
    areaSEEGP = None

    for page_idx in range(len(pdf)):
        page = pdf[page_idx]

        y1check = []
        y0check = []
        startpage = []
        endpage = []

        if areaUOP is None:
            for keyword in keywordsUOP:
                start = page.search_for(keyword)
                if start:
                    areaUOP = (page_idx, start[0])
                    startpage.append(page_idx)
                    for rect in start:
                        y1check.append(rect.y1)

        if areaSEEGP is None:
            for keyword in keywordsSEEGP:
                end = page.search_for(keyword)
                if end:
                    areaSEEGP = (page_idx, end[0])
                    endpage.append(page_idx)
                    for rect in end:
                        y0check.append(rect.y0)
        
        if len(startpage)>0 and len(endpage)>0:
            if startpage[0] == endpage[0]:
                for point1 in y1check:
                    for point0 in y0check:
                        if (point0 - point1) < 40 and point0 > point1:
                            areaUOP = None
                            areaSEEGP = None

    return areaUOP, areaSEEGP

In [12]:
# iterate pages in the pdf and extract preferrably tables or else words from UOP section

def page_scenario_and_extract(document, areaUOP, areaSEEGP):

    pdf = pymupdf.open(document)
    # language = language

    # inputs
    start_page_idx = areaUOP[0]
    end_page_idx = areaSEEGP[0]
    start_point = areaUOP[1].y1
    end_point = areaSEEGP[1].y0

    # outputs
    noTableMsg = []
    hasTableMsg = []
    hasDFMsg = []
    extractUOPwords = []

# iterate pages
    for page_idx in range(len(pdf)):
        page = pdf[page_idx]

# process four page scenarios, check for tables, extract tables else extract text as words
    # A: UOP all on a single page
        if page_idx == start_page_idx and page_idx == end_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[1] > start_point and bbox[3] < end_point:
                    tableAheader = tables[0].header.names
                    tableAdf = tables[0].to_pandas()
                hasTableMsg.append(tableAheader)
                hasDFMsg.append(tableAdf)
            else:
                noTableMsg.append('No tableA')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y0 > start_point and y1 < end_point:
                        extractUOPwords.append(text)

    # UOP across >1 page
    # B: current page is start page
        elif page_idx == start_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[1] > start_point:
                    tableBheader = tables[0].header.names
                    tableBdf = tables[0].to_pandas()
                hasTableMsg.append(tableBheader)
                hasDFMsg.append(tableBdf)
            else:
                noTableMsg.append('No tableB')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y0 > start_point:
                        extractUOPwords.append(text)

    # D: current page is neither start nor end page but is in the UOP area
        elif page_idx > start_page_idx and page_idx < end_page_idx:
            tables = page.find_tables(strategy = 'lines_strict')
            if tables.tables:
                tableDheader = tables[0].header.names
                tableDdf = tables[0].to_pandas()
                hasTableMsg.append(tableDheader)
                hasDFMsg.append(tableDdf)
            else:
                noTableMsg.append(page_idx)
                noTableMsg.append('No tableD')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    extractUOPwords.append(text)

    # C: current page is end page
        elif page_idx == end_page_idx:
            tables = page.find_tables()
            if tables.tables:
                bbox = tables[0].bbox
                if bbox[3] < end_point:
                    tableCheader = tables[0].header.names
                    tableCdf = tables[0].to_pandas()
                hasTableMsg.append(tableCheader)
                hasDFMsg.append(tableCdf)
            else:
                noTableMsg.append('No tableC')
                for word in page.get_text('words'):
                    x0, y0, x1, y1, text, *_ = word
                    if y1 < end_point:
                        extractUOPwords.append(text)

    return noTableMsg, hasTableMsg, hasDFMsg, extractUOPwords

    # noTableMsg lists page scenarios (and idx for scenario D) where no table is found
    # hasTableMsg lists the header row for found tables
    # hasDFMsg dataframe content in a list

In [13]:
# check extracted table is UOP table and extract Categories

def UOP_table_cats(hasDFMsg, hasTableMsg):

    # inputs
    hasDFMsg = hasDFMsg
    hasTableMsg = hasTableMsg

    # outputs
    uniqueCats = []
    tableInfo = []

    if len(hasDFMsg) >= 1:

    # this over simplifies by assuming the category is always in the first column
    # the header condition code oversimplifies by assuming table fragments split across pages without a header row contain no new category labels

        if 'Category' in hasTableMsg[0]:
            x = 'singular'
        elif 'Categories' in hasTableMsg[0]:
            x = 'plural'
        elif 'Eligible Project Category' in hasTableMsg[0]:
            x = 'phrase'
        else:
            x = 'not a UOP table or category not in 1st columns'

        DFname = hasDFMsg[0]
        if x == 'singular':
            uniqueC = DFname['Category'].unique()
            uniqueCats.append(uniqueC)
        elif x == 'plural':
            uniqueC = DFname['Categories'].unique()
            uniqueCats.append(uniqueC)
        elif x == 'phrase':
            uniqueC = DFname['Eligible Project Category'].unique()
            uniqueCats.append(uniqueC)
        else:
            tableInfo.append(x)

    if len(hasDFMsg) >1:
        tableInfo.append('More than one table found')
    if len(hasDFMsg) == 0:
        tableInfo.append('No table found')

    return tableInfo, uniqueCats

    # turn these into assert and proper error msgs later
        # if errorMsg is empty and uniqueCats contains a list of category like words, pdf has processed successfully
        # if errorMsg is not empty, there may be more than one table found, in which case DFname variable could be inaccurate
        # if errorMsg is not empty, the words Category or Categories may not be present in the header row, in which case may not be a UOP table or may be other words such as criteria


In [14]:
# process words into category word dataframes where token.similarity score passes threshold

def UOPwords_to_Catwords(extractUOPwords):

    # inputs
    extractUOPwords = extractUOPwords

    re = renewable_energy
    ee = energy_efficiency
    ppc = pollution_prevention_and_control
    esml = environmentally_sustainable_management_of_living_natural_resources_and_land_use
    tabc = terrestrial_and_aquatic_biodiversity_conservation
    ct = clean_transportation
    swwm = sustainable_water_and_wastewater_management
    cca = climate_change_adaptation
    ce = circular_economy_and_or_ecoefficient_projects
    gb = green_buildings

    similarity_threshold = 0.72

    # outputs
    simsre = {}
    simsee = {}
    simsppc = {}
    simsesml = {}
    simstabc = {}
    simsct = {}
    simsswwm = {}
    simscca = {}
    simsce = {}
    simsgb = {}


    # green_buildings
    for wordk in gb:
        dock = nlp(wordk)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if dock.similarity(docb) >= similarity_threshold:
                sim = dock.similarity(docb)
                simsgb[dock[0].text + ' ' + docb[0].text] = sim
                dfgb = pd.DataFrame.from_dict(simsgb, 'index')
            else:
                dfgb = 'is not gb'
    
    
    # renewable_energy
    for worda in re:
        doca = nlp(worda)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doca.similarity(docb) >= similarity_threshold:
                sim = doca.similarity(docb)
                simsre[doca[0].text + ' ' + docb[0].text] = sim
                dfre = pd.DataFrame.from_dict(simsre, 'index')
            else:
                dfre = 'is not re'


    # energy_efficiency
    for wordc in ee:
        docc = nlp(wordc)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docc.similarity(docb) >= similarity_threshold:
                sim = docc.similarity(docb)
                simsee[docc[0].text + ' ' + docb[0].text] = sim
                dfee = pd.DataFrame.from_dict(simsee, 'index')
            else:
                dfee = 'is not ee'
    
    # pollution_prevention_and_control
    for wordd in ppc:
        docd = nlp(wordd)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docd.similarity(docb) >= similarity_threshold:
                sim = docd.similarity(docb)
                simsppc[docd[0].text + ' ' + docb[0].text] = sim
                dfppc = pd.DataFrame.from_dict(simsppc, 'index')
            else:
                dfppc = 'is not ppc'

    # environmentally_sustainable_management_of_living_natural_resources_and_land_use
    for worde in esml:
        doce = nlp(worde)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doce.similarity(docb) >= similarity_threshold:
                sim = doce.similarity(docb)
                simsesml[doce[0].text + ' ' + docb[0].text] = sim
                dfesml = pd.DataFrame.from_dict(simsesml, 'index')
            else:
                dfesml = 'is not esml'

    # terrestrial_and_aquatic_biodiversity_conservation
    for wordf in tabc:
        docf = nlp(wordf)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docf.similarity(docb) >= similarity_threshold:
                sim = docf.similarity(docb)
                simstabc[docf[0].text + ' ' + docb[0].text] = sim
                dftabc = pd.DataFrame.from_dict(simstabc, 'index')
            else:
                dftabc = 'is not tabc'

    # clean_transportation
    for wordg in ct:
        docg = nlp(wordg)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docg.similarity(docb) >= similarity_threshold:
                sim = docg.similarity(docb)
                simsct[docg[0].text + ' ' + docb[0].text] = sim
                dfct = pd.DataFrame.from_dict(simsct, 'index')
            else:
                dfct = 'is not ct'

    # sustainable_water_and_wastewater_management
    for wordh in swwm:
        doch = nlp(wordh)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doch.similarity(docb) >= similarity_threshold:
                sim = doch.similarity(docb)
                simsswwm[doch[0].text + ' ' + docb[0].text] = sim
                dfswwm = pd.DataFrame.from_dict(simsswwm, 'index')
            else:
                dfswwm = 'is not swwm'

    # climate_change_adaptation
    for wordi in cca:
        doci = nlp(wordi)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if doci.similarity(docb) >= similarity_threshold:
                sim = doci.similarity(docb)
                simscca[doci[0].text + ' ' + docb[0].text] = sim
                dfcca = pd.DataFrame.from_dict(simscca, 'index')
            else:
                dfcca = 'is not cca'

    # circular_economy_and_or_ecoefficient_projects
    for wordj in ce:
        docj = nlp(wordj)
        for wordb in extractUOPwords:
            docb = nlp(wordb)
            if docj.similarity(docb) >= similarity_threshold:
                sim = docj.similarity(docb)
                simsce[docj[0].text + ' ' + docb[0].text] = sim
                dfce = pd.DataFrame.from_dict(simsce, 'index')
            else:
                dfce = 'is not ce'

    

    return simsre, simsee, simsppc, simsesml, simstabc, simsct, simsswwm, simscca, simsce, simsgb

In [191]:
# copy and adapt font scrape function from 'https://medium.com/@i190712/how-to-extract-text-from-pdf-on-the-basis-of-font-size-using-python-libraries-e931e583749b'

def fontInfo(language, document):
    
    if language == 'EN':
        keywordsUOP = ['Use of Proceeds', 'Use of Funds', 'Use of the Proceeds']
        keywordsSEEGP = ['Selection and Evaluation', 'Process for the Project Evaluation and Selection', 'Project Evaluation and Selection Process', 'Project Selection and Evaluation Process', 'Process for Project Evaluation and Selection', 'Project Selection and Assessment Process', 'Project Selection Criteria', 'Project evaluation & selection', 'Project Selection Process']
        keywordsSEEGP_test = ['Selection and Evaluation', 'Process for', 'Evaluation and Selection', 'Project Evaluation', 'Assessment Process', 'Selection process', 'Project selection criteria']


    results_UOP = []
    results_SEEGP = []
    
    pdf = pymupdf.open(document) 

    for page_idx in range(len(pdf)): 
        page = pdf[page_idx]
        dict = page.get_text("dict")
        blocks = dict["blocks"] 
        for block in blocks:
            if "lines" in block.keys():
                spans = block['lines']
                for span in spans:
                    data = span['spans']
                    for lines in data:
                        for keyword in keywordsUOP:
                            if keyword.lower() in lines['text'].lower().strip():
                                results_UOP.append((lines['text'], lines['size'], lines['bbox'], page_idx))
                        for keyword in keywordsSEEGP_test:
                            if keyword.lower() in lines['text'].lower().strip():
                                results_SEEGP.append((lines['text'], lines['size'], lines['bbox'], page_idx))

                            

    return results_UOP, results_SEEGP
                            


In [192]:
def find_max_font(results_UOP, results_SEEGP):
    
    max_font_size = 0
    for result_idx in range(len(results_UOP)):
        result = results_UOP[result_idx]
        if result[1] > max_font_size:
            max_font_size = result[1]
            max_idx_UOP = result_idx

    max_font_size = 0        
    for result_idx in range(len(results_SEEGP)):
        result = results_SEEGP[result_idx]
        if result[1] > max_font_size:
            max_font_size = result[1]
            max_idx_SEEGP = result_idx

    return max_idx_UOP, max_idx_SEEGP

In [193]:
def tocCheck(results_UOP, results_SEEGP, max_idx_UOP, max_idx_SEEGP):
    
    areaUOP = results_UOP[max_idx_UOP]
    areaSEEGP = results_SEEGP[max_idx_SEEGP]
    areaUOP_page = areaUOP[3]
    areaSEEGP_page = areaSEEGP[3]
    areaUOP_coords = areaUOP[2]
    areaSEEGP_coords = areaSEEGP[2]
    areaUOP_y1 = areaUOP_coords[3]
    areaSEEGP_y0 = areaSEEGP_coords[1]

    if areaUOP_page == areaSEEGP_page:
        if 0 < areaSEEGP_y0 - areaUOP_y1 < 40:
            results_UOP_remove = results_UOP.pop(max_idx_UOP)
            results_SEEGP_remove = results_SEEGP.pop(max_idx_SEEGP)

            max_idx_UOP, max_idx_SEEGP = find_max_font(results_UOP, results_SEEGP)

            areaUOP = results_UOP[max_idx_UOP]
            areaSEEGP = results_SEEGP[max_idx_SEEGP]
            areaUOP_page = areaUOP[3]
            areaSEEGP_page = areaSEEGP[3]
            areaUOP_coords = areaUOP[2]
            areaSEEGP_coords = areaSEEGP[2]
            areaUOP_y1 = areaUOP_coords[3]
            areaSEEGP_y0 = areaSEEGP_coords[1]

    return areaUOP_page, areaUOP_y1, areaSEEGP_page, areaSEEGP_y0

In [201]:
# run the font function
# 

document = '/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/refinement_reports/BTG_Sustainable_Financing_Framework.pdf'
language = 'EN'
results_UOP, results_SEEGP = fontInfo(language, document)
max_idx_UOP, max_idx_SEEGP = find_max_font(results_UOP, results_SEEGP)
print(results_UOP, max_idx_UOP)
print(results_SEEGP, max_idx_SEEGP)
#results_UOP_remove = results_UOP.pop(max_idx_UOP)
#results_SEEGP_remove = results_SEEGP.pop(max_idx_SEEGP)
areaUOP_page, areaUOP_y1, areaSEEGP_page, areaSEEGP_y0 = tocCheck(results_UOP, results_SEEGP, max_idx_UOP, max_idx_SEEGP)

print(results_UOP)
print(results_SEEGP)
print(areaUOP_page, areaUOP_y1)
print(areaSEEGP_page, areaSEEGP_y0)


[('Use of Proceeds', 13.0, (241.72439575195312, 546.1080932617188, 339.0540771484375, 561.994140625), 1), ('Bond Guidelines 2021 (“SBG”) (Use of Proceeds, Pro\xad', 10.0, (39.96849822998047, 593.0523071289062, 286.2991943359375, 605.2722778320312), 20), ('3.1 Use of Proceeds', 16.0, (308.9764099121094, 60.266231536865234, 455.1044921875, 80.4582290649414), 20), ('are compliant with the current Use of Proceeds.', 10.0, (308.9764099121094, 254.1357421875, 524.723388671875, 266.3557434082031), 31)] 2
[('Process for Project Selection and Evaluation', 13.0, (241.72439575195312, 572.1080932617188, 501.7372131347656, 587.994140625), 1), ('Process for Project Selection and Evaluation', 13.0, (241.72439575195312, 572.1080932617188, 501.7372131347656, 587.994140625), 1), ('cess for Project Evaluation and Selection, Management ', 10.0, (39.96849822998047, 607.0523071289062, 288.9834899902344, 619.2722778320312), 20), ('cess for Project Evaluation and Selection, Management ', 10.0, (39.96849822998

In [131]:
print(results)

print(len(results))

[('Use of proceeds ', 11.0, (360.3396911621094, 494.72601318359375, 439.4066162109375, 507.93701171875), 3), ('green portfolio (see section a. Use of Proceeds). ', 9.699999809265137, (312.0, 192.61102294921875, 529.4946899414062, 204.26072692871094), 14), ('a.\t Use of proceeds', 9.699999809265137, (312.0, 335.6109924316406, 390.5499267578125, 347.2607116699219), 14), ('a. Use of proceeds', 12.0, (312.0, 463.9993896484375, 421.3788757324219, 479.13140869140625), 14), ('Use of Proceeds – Eligible Green Expenditures. The ', 9.699999809265137, (312.0, 504.61102294921875, 529.47412109375, 516.2607421875), 19), ('established in section a. Use of Proceeds, the net ', 9.699999809265137, (85.03939819335938, 439.6109924316406, 302.5270690917969, 451.2607116699219), 20), ('an annual report on the use of proceeds that shall ', 9.699999809265137, (85.03939819335938, 244.61102294921875, 302.5162353515625, 256.2607421875), 21)]
7


In [50]:
# the visual on the pymupdf textpage page, about half way down under structure of dictionary outputs
# page dicts contain a list of blocks
# a (text) block dict containes a lines key ... which contains the lines in the block
# the lines key contains a list of spans keys ...
# the spans key contains font info, text info and bbox info

print(results[0])
print(results[1])
#print(results[2])
#print(results[3])
#print(results[4])

print(results_data[0]) #  just the span_key values
print(results_data[1])
#print(results_data[2])
#print(results_data[3])
#print(results_data[4])

print(results_span[0]) #just the lines containing UOP
print(results_span[1])
#print(results_span[2])
#print(results_span[3])
#print(results_span[4])

print(results_block[0]) #all the lines in the first block that UOP is in
print(results_block[1])
#print(results_block[2])
#print(results_block[3])
#print(results_block[4]) #all the lines in the last block that UOP is in


print(len(results))



('Use of proceeds', 12.0, (46.69000244140625, 307.8548278808594, 131.768798828125, 322.25482177734375), 14)
('Use of Proceeds', 15.0, (36.25, 387.0630187988281, 152.7864990234375, 406.3380432128906), 14)
[{'size': 12.0, 'flags': 4, 'bidi': 0, 'char_flags': 16, 'font': 'Moderat-Light', 'color': 4296670, 'alpha': 255, 'ascender': 0.9300000071525574, 'descender': -0.27000001072883606, 'text': '1. ', 'origin': (36.25, 319.01483154296875), 'bbox': (36.25, 307.8548278808594, 46.69000244140625, 322.25482177734375)}, {'size': 12.0, 'flags': 4, 'bidi': 0, 'char_flags': 16, 'font': 'Moderat-Light', 'color': 4210752, 'alpha': 255, 'ascender': 0.9300000071525574, 'descender': -0.27000001072883606, 'text': 'Use of proceeds', 'origin': (46.69000244140625, 319.01483154296875), 'bbox': (46.69000244140625, 307.8548278808594, 131.768798828125, 322.25482177734375)}]
[{'size': 15.0, 'flags': 20, 'bidi': 0, 'char_flags': 24, 'font': 'Moderat-Black', 'color': 4296670, 'alpha': 255, 'ascender': 0.93199998140

In [26]:
# run the program
# notable

document = '/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/refinement_reports/BTG_Pactual_Green,_Social_and_Sustainable_Financing_Framework.pdf'
language = 'EN'
areaUOP, areaSEEGP = find_uop(document, language)


In [27]:
print(areaUOP)
print(areaSEEGP)

(12, Rect(358.7259826660156, 387.7625732421875, 441.8499450683594, 402.1625671386719))
(14, Rect(48.7660026550293, 323.850830078125, 271.7727966308594, 338.2508239746094))


In [19]:
# collate the outputs
# UOP table,  0% ES

print(filepath.name, type(filepath.name))
print(language, type(language))
print(areaUOP[0], type(areaUOP[0]))
print(areaSEEGP[0], type(areaSEEGP[0]))
if len(hasDFMsg) != 0:
    if len(tableInfo) == 1:   
        print(tableInfo[0], type(tableInfo[0]))
    elif len(tableInfo) == 2:
        print(tableInfo[1], type(tableInfo[1]))
    elif len(uniqueCats) > 0:
        print(uniqueCats, len(uniqueCats))
    else:
        print('no categories from tables')
if len(uniqueCats) == 0:
    print(f"renewable_energy  {simsre}")
    print(f"energy_efficiency {simsee}")
    print(f"pollution_prevention_and_control {simsppc}")
    print(f"environmentally_sustainable_management_of_living_natural_resources_and_land_use {simsesml}")
    print(f"terrestrial_and_aquatic_biodiversity_conservation {simstabc}")
    print(f"clean_transportation {simsct}")
    print(f"sustainable_water_and_wastewater_management {simsswwm}")
    print(f"climate_change_adaptation {simscca}")
    print(f"circular_economy_and_or_ecoefficient_projects {simsce}")
    print(f"green_buildings {simsgb}")
else:
    print('categories from tables')

BTG_Pactual_Green,_Social_and_Sustainable_Financing_Framework.pdf <class 'str'>
EN <class 'str'>
1 <class 'int'>
1 <class 'int'>
renewable_energy  {}
energy_efficiency {}
pollution_prevention_and_control {}
environmentally_sustainable_management_of_living_natural_resources_and_land_use {}
terrestrial_and_aquatic_biodiversity_conservation {}
clean_transportation {}
sustainable_water_and_wastewater_management {}
climate_change_adaptation {}
circular_economy_and_or_ecoefficient_projects {}
green_buildings {}
